# FIN-02 | Demo — Customer Churn Prediction

## 1. Setup
This notebook provides a clean inference pipeline to demonstrate predicting customer churn using our trained model. It covers single predictions, batch predictions, and input validation handling.

## 2. Install Dependencies

In [ ]:
# Uncomment in Colab:
# !pip install -r requirements.txt
import sys, os
sys.path.insert(0, os.path.abspath('.'))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


## 3. Load Saved Model

In [ ]:
from src.predict import load_model, predict_churn

model, feature_cols = load_model()
print(f'Model loaded. Expects {len(feature_cols)} features.')
print('Features:', feature_cols)


## 4. What the Model Predicts
- **Churn Definition:** A customer is considered churned if their transaction activity drops off completely during the labeling window.
- **Observation Window:** The model observes history up to a specific cutoff date to build features.
- **Risk Tiers:** 
  - High Risk: prob > 0.7
  - Medium Risk: 0.3 <= prob <= 0.7
  - Low Risk: prob < 0.3

## 5. Example Predictions

In [ ]:
examples = [
    {"account_id": 1, "trans_count": 5, "trans_freq_monthly": 1.0, "avg_balance": 100.0, "tenure_months": 12},  # High risk
    {"account_id": 2, "trans_count": 50, "trans_freq_monthly": 5.0, "avg_balance": 1000.0, "tenure_months": 36}, # Medium risk
    {"account_id": 3, "trans_count": 200, "trans_freq_monthly": 15.0, "avg_balance": 15000.0, "tenure_months": 60} # Low risk
]

probs = []
for ex in examples:
    # Ensure all feature_cols are present
    features = {col: ex.get(col, 0) for col in feature_cols}
    res = predict_churn(features)
    probs.append(res['churn_probability'])
    print(f"Account {ex['account_id']} -> Prob: {res['churn_probability']:.2f} | Tier: {res['risk_tier']} | Action: {res['recommended_action']}")

plt.figure(figsize=(6, 4))
sns.barplot(x=['High Risk Profile', 'Medium Risk Profile', 'Low Risk Profile'], y=probs)
plt.title('Churn Probabilities for Examples')
plt.ylabel('Probability')
plt.show()


## 6. Batch Prediction Example

In [ ]:
df_batch = pd.DataFrame(examples * 2) # just duplicating to make 6 rows
df_batch['account_id'] = range(101, 107)

batch_res = []
for _, row in df_batch.iterrows():
    features = {col: row.get(col, 0) for col in feature_cols}
    res = predict_churn(features)
    batch_res.append(res)
    
res_df = pd.DataFrame(batch_res)
display(pd.concat([df_batch['account_id'], res_df], axis=1))


## 7. Input Validation Demo

In [ ]:
print("Testing with missing features...")
bad_features = {"trans_count": 10} # Missing many required cols
try:
    predict_churn(bad_features)
except Exception as e:
    print("Gracefully caught exception:", e)


## 8. Limitations & Responsible AI

- **Educational use only:** This is a demonstration project.
- **Self-defined churn label:** It acts as a proxy, not an absolute ground truth.
- **Historical Dataset:** Based on a 1990s Czech bank dataset — not representative of modern digital banking behaviors.
- **Not for automation:** Model is not intended for automatic account closure decisions.
- **Human review:** Always required before taking any retention or punitive action.